# 11 — Bài Q1: backbone da liễu + Hybrid Metadata Fusion + prior cân bằng (Colab A100)

Chạy `experiments/stage9_pretrain_hybrid.yaml` — **7 cấu hình × 5 fold = 35 run**, rồi hậu xử lý label shift và bảng so sánh
với mốc **M0** (Swin-B-384 · MoE-D, test 0.519 — đã có từ stage 6, *không train lại*).

Chung cho mọi run (recipe M0): ảnh clin + derm + metadata · overlay `r5_nobce` (CE, squash, aug mạnh, drop path 0.1) ·
MoE `d_model 256` · 20 epoch, patience 5 · batch 16 × 4 = 64 · bf16. Mỗi cấu hình đổi **đúng 1 yếu tố** so với mốc của nó.

| # | Run (`<backbone>__abl_<variant>`) | Đổi gì | So với |
|---|---|---|---|
| 1 | `vit_base__abl_D` | MoE-D trên ViT-B/224 ImageNet | đối chứng cho 2, 3 |
| 2 | `panderm_base__abl_D` | backbone → **PanDerm-Base** | 1 |
| 3 | `dermlip_vit_b16__abl_D` | backbone → **DermLIP** | 1 |
| 4 | `panderm_base__abl_HMF` | + **HMF** | 2 |
| 5 | `dermlip_vit_b16__abl_HMF` | + **HMF** | 3 |
| 6 | `swin_base_384__abl_HMF` | + **HMF** | M0 |
| 7 | `swin_base_384__abl_q1` | sampler q0.5 → **q1.0** | M0 |

---
### Ghi chú kiến trúc (đọc sơ để hiểu từng method)

**MoE-D (mốc M0, đã có).** Mỗi ảnh qua backbone dùng chung → vector đặc trưng → chiếu 256 chiều. Metadata tách 2 token
(demo: tuổi/giới/vị trí/tông da; MONET: 14 điểm khái niệm). 4 token `{clin, derm, demo, monet}` → mỗi token một expert MLP,
**gate** theo mẫu quyết định trộn bao nhiêu từ mỗi nguồn → 3 head long-tail (logit-adjusted τ = 0 / 0.5 / 1), trung bình
xác suất. Thêm head phụ cho từng token và modality dropout (che ngẫu nhiên 1 nguồn khi train).
```
clin ─ backbone ─ f_c ┐
derm ─ backbone ─ f_d ┼─ 4 token ─ expert + gate ─ 3 head long-tail ─ p
demo ─ MLP ──── t_m   ┤
MONET ─ MLP ─── t_M   ┘
```

**PanDerm-Base (run 2, 4).** ViT-B/16 (86M) pretrain trên ~2 triệu ảnh da liễu (dermoscopy, clinical, TBP, mô bệnh học) bằng
MAE + chưng cất CLIP. *Vì sao:* backbone hiện tại học trên ImageNet (ảnh đời thường); nhóm lớp đang mất điểm nhất
(AKIEC / SCCKA / BKL bị nhầm sang BCC) khác nhau ở chi tiết bề mặt da mà mô hình da liễu đã học sẵn. Một entry leaderboard
chỉ đổi sang PanDerm-Base (không metadata) đã đạt 0.546 > 0.519. Kiến trúc giữ nguyên MoE-D, chỉ thay backbone.

**DermLIP (run 3, 5).** ViT-B/16 CLIP ảnh–văn bản trên Derm1M (~400k cặp ảnh da liễu + mô tả, >390 bệnh). Đối chứng thứ hai:
nếu cả PanDerm và DermLIP cùng hơn ViT ImageNet (run 1) thì lợi ích là của *pretrain da liễu*, không phải may của 1 mô hình.
Ba backbone ViT-B/224 dùng chung **layer decay 0.75** (lr nhỏ dần về các block đầu — chuẩn khi fine-tune MAE/CLIP).

**HMF — Hybrid Metadata Fusion (run 4, 5, 6).** Metadata vào ở **3 độ sâu** trong *một* mô hình:
```
              backbone ─→ f_c, f_d (vector)  và  P_c, P_d (lưới patch)
metadata m ─┬─ (1) SỚM  : [f_c ‖ f_d ‖ m] → MLP                                   → logits_E
            ├─ (2) GIỮA : m → query → cross-attention lên P_c, P_d (MGSP) → token q_c, q_d
            └─ (3) TOKEN: {f_c, f_d, demo, monet, q_c, q_d} → expert + gate + 3 head  → p_T
đầu ra = ½ (log softmax(logits_E) + log p_T)       loss = CE(đầu ra) + 0.25·[CE(E) + loss(T)] + head phụ
```
*Vì sao:* ở stage 8, nối metadata thẳng (R2) mạnh ở MEL/BCC, còn metadata làm query (R4) mạnh ở INF/NV/DF — hai cách sai ở
những ca khác nhau (~12%). HMF gom cả hai vào cùng mô hình; trộn 2 nhánh bằng trung bình cố định (không thêm gate).

**q1 — prior cân bằng (run 7).** Giữ nguyên M0, chỉ đổi sampler q 0.5 → 1.0 (mỗi lớp được lấy mẫu đều khi train).
*Vì sao:* test lệch phân bố lớp mạnh so với train (BCC 48% → ~13%; AKIEC, SCCKA, BKL tăng 2–2.7 lần), mô hình hiện
"nghiện" BCC (dự đoán ~145 ca BCC trên test). Train với prior trung tính rồi để hậu xử lý cắm prior test là cách đúng theo
lý thuyết label shift.

**Hậu xử lý BBSE (bước 7, không train).** Ước lượng phân bố lớp của test *chỉ từ xác suất dự đoán trên ảnh test* (không nhãn):
học "kiểu nhầm" của mô hình trên OOF (có nhãn train), rồi giải ngược tỉ lệ lớp sao cho khớp tổng dự đoán trên test; sau đó
nhân lại prior p'(y|x) ∝ p(y|x)·π_test/π_train. Mỗi mô hình chọn `raw` / `bbse` / `bbse½` bằng **mô phỏng cross-fit trên OOF**
(chốt trước khi nộp) → **1 file nộp / cấu hình**. Không dùng điểm hay nhãn test để chọn.

---
**Thời gian (ước lượng, A100 80GB, 3 run song song):** ViT-B/224 ~30 phút/run, Swin-B/384 ~50 phút/run khi chạy một mình
→ 1 máy ~11–12 giờ, **2 máy (`HUONG="top"` + `"down"`) ~6 giờ**. Số đo thật in ra sau run đầu tiên.

**Trước khi chạy (1 lần):**
1. DermLIP là repo HuggingFace *gated*: đăng nhập HF → mở https://huggingface.co/redlessone/DermLIP_ViT-B-16 → **Request access**
   (duyệt tự động). Tạo token (Settings → Access Tokens, quyền read) → Colab: biểu tượng 🔑 *Secrets* → thêm `HF_TOKEN`, bật
   *Notebook access*. Không có token thì bỏ `dermlip_vit_b16` khỏi `BACKBONES`.
2. Runtime → **A100 GPU**, bật High-RAM → **Run all**. Ngắt giữa chừng: mở lại, Run all — run đã xong (trên Drive) được bỏ qua.
3. Máy thứ 2: mở bản sao notebook, đặt `HUONG = "down"`.

Giấy phép: PanDerm và DermLIP chỉ dùng cho nghiên cứu phi thương mại; **không công khai checkpoint đã fine-tune** (DermLIP cấm
phân phối lại). Notebook không lưu checkpoint (`keep_checkpoints: false`).

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DRIVE_DIR = "/content/drive/MyDrive/milk10k_stage9"           # thư mục riêng của stage 9
M0_DIR = "/content/drive/MyDrive/milk10k_ablation/outputs"    # run M0 (stage 6) để so sánh — chỉ đọc

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage9_pretrain_hybrid.yaml"
BACKBONES = ["vit_base", "panderm_base", "dermlip_vit_b16", "swin_base_384"]   # bỏ dermlip_vit_b16 nếu không có HF_TOKEN
VARIANTS = ["D", "HMF", "q1"]       # mỗi variant chỉ chạy trên các backbone ghi trong yaml
FOLDS = None                        # None = 5 fold | vd [0]: chạy thử fold 0 trước

HUONG = "top"                       # top: từ trên xuống | down: từ dưới lên (máy thứ 2)
TIME_BUDGET_H = 23.0                # không bắt đầu run mới nếu không kịp xong trước mốc này
JOBS_PER_GPU = "auto"               # auto: 3 nếu VRAM >= 70GB, 2 nếu >= 38GB | số cụ thể (giảm nếu CUDA out of memory)

In [ ]:
# ===================== THAM SỐ TRAIN (giống M0 stage 6) =====================
PARAMS = {
    "train.batch_size": 16,             # 16 lesion = 32 ảnh mỗi bước; CUDA out of memory -> 8 và accum_steps 8
    "train.accum_steps": 4,             # batch hiệu dụng 64
    "model.grad_checkpointing": False,
    "train.amp_dtype": "bf16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Google Drive, GPU và số run song song

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/outputs", exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> A100 GPU"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
if "A100" not in props.name:
    print(f"CẢNH BÁO: đang dùng {props.name}, không phải A100. bf16 cần GPU Ampere trở lên (A100, L4, H100).")
jobs = JOBS_PER_GPU if JOBS_PER_GPU != "auto" else (3 if vram >= 70 else 2 if vram >= 38 else 1)
cpus = os.cpu_count()
PARAMS["data.num_workers"] = max(2, cpus // jobs - 1)
GPUS = ",".join(["0"] * jobs)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # giảm phân mảnh VRAM
print(f"{props.name} {vram:.0f}GB | {cpus} CPU | {jobs} run song song | {PARAMS['data.num_workers']} worker/run")

## 2. Code, dữ liệu, token HuggingFace

In [ ]:
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/content/MILK10k"            # ổ SSD của Colab: đọc ảnh nhanh hơn Drive
os.environ["MILK10K_ROOT"] = DATA_DIR
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" gdown openpyxl safetensors huggingface_hub
!git log -1 --format="Commit: %h %s"
!python scripts/setup_data.py --drive-url "{DRIVE_URL}" --dest {DATA_DIR}

In [ ]:
# Trọng số pretrain lưu trên Drive (tải 1 lần, máy thứ 2 dùng lại). HF_TOKEN lấy từ Colab Secrets.
os.environ["MILK10K_WEIGHTS"] = DRIVE_DIR + "/weights"
os.makedirs(os.environ["MILK10K_WEIGHTS"], exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Đã nạp HF_TOKEN từ Colab Secrets")
except Exception as e:
    print("Không có HF_TOKEN (", type(e).__name__, ") -> DermLIP sẽ không tải được")

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline, gồm HMF + kiến trúc PanDerm/DermLIP (~3 phút, không tải trọng số)
!python -m pytest tests -q -p no:cacheprovider

In [ ]:
# Tải + kiểm tra trọng số pretrain trước khi train: nạp PHẢI khớp 100% khoá (lỗi thì dừng ở đây, không lãng phí GPU)
need = [b for b in ("panderm_base", "dermlip_vit_b16") if b in BACKBONES]
for b in need:
    !python -c "import sys; sys.path.insert(0,'src'); from milk10k.models.derm_backbones import create_derm_backbone as c; e=c('{b}',224,True); print('{b}: OK', sum(p.numel() for p in e.parameters())//10**6, 'M tham số, mean', e.pretrained_cfg['mean'])"

## 3. Đồng bộ kết quả lên Drive và danh sách run

In [ ]:
# Chạy tiếp: run đã xong trên Drive được chép về outputs/ và bỏ qua. --shared-dir: 2 máy không chạy trùng run.
restore_arg = (f"--restore {DRIVE_DIR}/outputs --shared-dir {DRIVE_DIR}/outputs"
               + (" --reverse" if HUONG == "down" else ""))
GPU_LOG = f"outputs/_gpu_usage_{HUONG}.csv"
os.makedirs("outputs", exist_ok=True)
# Đồng bộ nền mỗi 5 phút (bỏ checkpoint .pt) + ghi mức dùng GPU mỗi phút
sync_cmd = f"while true; do rsync -a --exclude '*.pt' outputs/ '{DRIVE_DIR}/outputs/'; sleep 300; done"
if "sync_proc" not in globals() or sync_proc.poll() is not None:
    sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
    gpu_log = subprocess.Popen("nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw "
                               f"--format=csv -l 60 > {GPU_LOG}", shell=True)
print("Đồng bộ nền:", DRIVE_DIR + "/outputs")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--backbones {','.join(BACKBONES)} --variants {','.join(VARIANTS)}"
              + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else ""))
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus {GPUS} --est-run-h 0.7 --time-budget-h {TIME_BUDGET_H}

## 5. Hậu xử lý label shift + bảng so sánh (CPU, vài phút)
Chạy 2 máy: đợi cả hai xong rồi chạy ở một máy. Kết quả: `DRIVE_DIR/outputs/posthoc_shift/`
- `posthoc_shift.md` / `.xlsx`: Dice 5 fold, Dice mô phỏng test (raw / bbse / bbse½), cách đã chọn, Δ so với M0
- `submissions/<run>__cv5__<cách>/submission.csv`: **file nộp leaderboard, 1 file / cấu hình** (thư mục `_khac/` không nộp)

In [ ]:
from IPython.display import Markdown, display
!rsync -a --exclude '*.pt' outputs/ "{DRIVE_DIR}/outputs/"
!python scripts/posthoc_shift.py --dirs "{DRIVE_DIR}/outputs" "{M0_DIR}" --out outputs/posthoc_shift --ref swin_base_384__abl_M0 > /dev/null
display(Markdown(open("outputs/posthoc_shift/posthoc_shift.md", encoding="utf-8").read()))
!rsync -a outputs/posthoc_shift/ "{DRIVE_DIR}/outputs/posthoc_shift/"
print("File nộp:", DRIVE_DIR + "/outputs/posthoc_shift/submissions/")

In [ ]:
# Mức dùng GPU trong lúc train (lấy mẫu mỗi phút)
import pandas as pd
u = pd.read_csv(GPU_LOG, skipinitialspace=True)
util = u.filter(like="utilization").iloc[:, 0].str.rstrip(" %").astype(float)
mem = u.filter(like="memory.used").iloc[:, 0].str.rstrip(" MiB").astype(float) / 1024
print(f"GPU util trung bình {util.mean():.0f}% (trung vị {util.median():.0f}%), VRAM tối đa {mem.max():.1f}GB, {len(u)} mẫu")